# 02 物理成功评估与视频复核

        这一节解决一个关键问题：日志里的 success 是否真的代表机器人夹起杯子并放到盘子上。大家会看到为什么要把旧的几何成功条件和 `physical_success` 分开统计，并用视频关键帧复核成功与失败。


In [ ]:
from pathlib import Path
import json
import os
import shutil
import subprocess
import sys


def find_topic_root():
    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents]:
        if (candidate / "assets" / "metrics_snapshot.json").exists():
            return candidate
    raise RuntimeError("请从 AMD ROCm 专题目录或 notebooks 子目录启动 Jupyter。")


TOPIC_ROOT = find_topic_root()
ASSET_DIR = TOPIC_ROOT / "assets"
NOTEBOOK_DIR = TOPIC_ROOT / "notebooks"
PROJECT_ROOT = Path(os.environ.get("PROJECT_ROOT", "/path/to/every-embodied/mujoco_pnp"))
DATA_ROOT = Path(os.environ.get("DATA_ROOT", "/path/to/datasets/every_embodied"))
OUTPUT_ROOT = Path(os.environ.get("OUTPUT_ROOT", TOPIC_ROOT / "outputs"))
MODEL_ROOT = Path(os.environ.get("MODEL_ROOT", PROJECT_ROOT / "ckpt"))

print("TOPIC_ROOT =", TOPIC_ROOT)
print("PROJECT_ROOT =", PROJECT_ROOT)
print("DATA_ROOT =", DATA_ROOT)
print("OUTPUT_ROOT =", OUTPUT_ROOT)
print("MODEL_ROOT =", MODEL_ROOT)


In [ ]:
try:
    from IPython.display import Image, Markdown, display
except Exception:
    class Markdown(str):
        pass

    def display(obj):
        print(obj)

    def Image(filename=None, width=None):
        return f"[image] {filename}"


def show_asset(filename, width=960):
    path = ASSET_DIR / filename
    if path.exists():
        display(Image(filename=str(path), width=width))
    else:
        print(f"缺少素材：{path}")


def md_table(headers, rows):
    lines = ["| " + " | ".join(headers) + " |", "| " + " | ".join(["---"] * len(headers)) + " |"]
    for row in rows:
        lines.append("| " + " | ".join(str(x) for x in row) + " |")
    display(Markdown("\n".join(lines)))


## Checkpoint 1：读取本专题的示例指标


In [ ]:
snapshot_path = ASSET_DIR / "metrics_snapshot.json"
snapshot = json.loads(snapshot_path.read_text(encoding="utf-8"))
print(json.dumps(snapshot["best_summary"], ensure_ascii=False, indent=2))


## Checkpoint 2：理解 `physical_success`


In [ ]:
def physical_success(row, min_lift=0.03, min_lift_steps=3, upright_threshold=0.7):
    """示例判定逻辑。实际项目中应以 eval 脚本记录的字段为准。"""
    legacy = bool(row.get("legacy_success", row.get("success", False)))
    max_lift = float(row.get("max_target_lift", row.get("max_mug_lift", 0.0)))
    lifted_steps = int(row.get("lifted_steps", 0))
    upright = float(row.get("final_target_upright_cos", row.get("final_mug_upright_cos", 1.0)))
    return legacy and max_lift >= min_lift and lifted_steps >= min_lift_steps and upright >= upright_threshold


examples = [
    {"name": "几何成功但倒杯", "success": True, "max_mug_lift": 0.08, "lifted_steps": 80, "final_mug_upright_cos": 0.2},
    {"name": "推到盘边但没夹起", "success": True, "max_mug_lift": 0.005, "lifted_steps": 0, "final_mug_upright_cos": 0.99},
    {"name": "真抓取并直立放置", "success": True, "max_mug_lift": 0.09, "lifted_steps": 120, "final_mug_upright_cos": 0.96},
]
md_table(["样例", "physical_success"], [(e["name"], physical_success(e)) for e in examples])


旧 success 可以作为辅助指标，但不能替代视频和物理状态复核。特别是抓杯任务里，推、挤、倒杯都可能让终态几何看起来接近成功。


## Checkpoint 3：复核成功与失败关键帧


In [ ]:
show_asset("smolvla_blue_failure_sequence.jpg", width=1100)
show_asset("smolvla_blue_success_sequence.jpg", width=1100)
show_asset("act_failure_sequence.jpg", width=1100)
show_asset("act_success_sequence.jpg", width=1100)


看视频时，大家要沿时间轴观察四件事：是否接触目标杯、是否稳定夹起、是否搬运到盘子上、终态是否直立。


## Checkpoint 4：批量 JSONL 统计模板


In [ ]:
def summarize_jsonl(path):
    path = Path(path)
    if not path.exists():
        print(f"文件不存在：{path}")
        return None
    rows = [json.loads(line) for line in path.read_text(encoding="utf-8").splitlines() if line.strip()]
    total = len(rows)
    legacy = sum(bool(r.get("success") or r.get("legacy_success")) for r in rows)
    physical = sum(bool(r.get("physical_success")) for r in rows)
    return {"total": total, "legacy_success": legacy, "physical_success": physical}


example_jsonl = OUTPUT_ROOT / "eval_result.jsonl"
print("把自己的 JSONL 放到这里后运行：", example_jsonl)
print(summarize_jsonl(example_jsonl))
